## Calculating the Intersections of Subjects Across fMRI and sMRI

In [23]:
import os
import glob
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

dir_path = r"C:\Users\Faruk\Code\CCIR_Project"
RANDOM_STATE = 17
AGE_TEST_SIZE = 0.1
PSY_TEST_SIZE = 0.33

---
## NKI

**Protocol**: acq-645, ses-BAS1, CC200 atlas, reg-36Parameter, PartialNilearn correlations

**Intersection**: FreeSurfer subjects & CPAC subjects (file on disk) & demographic label available

In [24]:
# Sessions: BAS1 (baseline), FLU1/FLU2 (follow-ups at different ages), BAS2 (second baseline)
# TRT excluded: same-wave reliability rescan, age label identical to FLU1/FLU2, no new developmental timepoint
NKI_SESSIONS = ["BAS1", "FLU1", "FLU2", "BAS2"]

# For each session: subjects with acq-645 FC (CC200 proxy) AND FreeSurfer sMRI on disk
nki_both_per_session = {}
for ses in NKI_SESSIONS:
    fs_dirs = glob.glob(rf"{dir_path}\NKI_FreeSurfer\freesurfer\sub-*_ses-{ses}")
    fs_subs = {os.path.basename(d).split("_")[0].replace("sub-", "") for d in fs_dirs}

    cpac_files = glob.glob(
        rf"{dir_path}\NKI_CPAC\cpac_RBCv0\sub-*\ses-{ses}\func"
        rf"\*_task-rest_acq-645_atlas-CC200_space-MNI152NLin6ASym_reg-36Parameter_desc-PartialNilearn_correlations.tsv"
    )
    cpac_subs = {os.path.basename(f).split("_")[0].replace("sub-", "") for f in cpac_files}

    both = fs_subs & cpac_subs
    nki_both_per_session[ses] = both
    print(f"ses-{ses}: FreeSurfer={len(fs_subs)}  CPAC(acq-645)={len(cpac_subs)}  both={len(both)}")

ses-BAS1: FreeSurfer=1075  CPAC(acq-645)=815  both=810
ses-FLU1: FreeSurfer=334  CPAC(acq-645)=272  both=269
ses-FLU2: FreeSurfer=186  CPAC(acq-645)=86  both=86
ses-BAS2: FreeSurfer=78  CPAC(acq-645)=71  both=71


In [25]:
# NKI demographics: one row per (participant_id, session_id)
nki_demo = pd.read_csv(rf"{dir_path}\NKI_BIDS\study-NKI_desc-participants.tsv", sep="\t")
nki_demo["participant_id"] = nki_demo["participant_id"].astype(str)

# Build age pool across all sessions — ID stored as "{sub}-{ses}" to encode session
nki_age_records = []
for ses in NKI_SESSIONS:
    ses_rows = nki_demo[nki_demo["session_id"] == ses].copy()
    ses_rows = ses_rows[ses_rows["participant_id"].isin(nki_both_per_session[ses])]
    ses_rows = ses_rows[ses_rows["age"].notna()]
    ses_rows["subject_session_id"] = ses_rows["participant_id"] + "-" + ses
    nki_age_records.append(ses_rows[["subject_session_id", "age"]])

nki_age_df = pd.concat(nki_age_records, ignore_index=True)
nki_age_pool = nki_age_df["subject_session_id"].tolist()
print(f"NKI age pool (all sessions, FS & CPAC & age not NaN): {len(nki_age_pool)}")
for ses in NKI_SESSIONS:
    n = sum(1 for s in nki_age_pool if s.endswith(f"-{ses}"))
    print(f"  ses-{ses}: {n}")

NKI age pool (all sessions, FS & CPAC & age not NaN): 1236
  ses-BAS1: 810
  ses-FLU1: 269
  ses-FLU2: 86
  ses-BAS2: 71


In [27]:
nki_age_trainval, nki_age_test = train_test_split(
    sorted(nki_age_pool), test_size=AGE_TEST_SIZE, random_state=RANDOM_STATE
)
print(f"NKI age  - trainval: {len(nki_age_trainval)}, test: {len(nki_age_test)}")

# Psy pool: BAS1 only (p_factor is a subject-level measure, not session-specific)
bas1_rows = nki_demo[nki_demo["session_id"] == "BAS1"].copy()
bas1_rows = bas1_rows[bas1_rows["participant_id"].isin(nki_both_per_session["BAS1"])]
nki_psy_pool = (
    bas1_rows[bas1_rows["p_factor_mcelroy_harmonized_all_samples"].notna()]["participant_id"]
    .apply(lambda s: f"{s}-BAS1")
    .tolist()
)
nki_psy_trainval, nki_psy_test = train_test_split(
    sorted(nki_psy_pool), test_size=PSY_TEST_SIZE, random_state=RANDOM_STATE
)
print(f"NKI psy  - trainval: {len(nki_psy_trainval)}, test: {len(nki_psy_test)}")

NKI age  - trainval: 1112, test: 124
NKI psy  - trainval: 111, test: 55


---
## NKI_AgeTrimmed

**Protocol**: acq-645, ses-BAS1, CC200 atlas, reg-36Parameter, PartialNilearn correlations

**Intersection**: FreeSurfer subjects & CPAC subjects (file on disk) & demographic label available & age <= 20

In [28]:
# NKItrimmed reuses nki_both_per_session and nki_demo from the NKI cells above
# Same acq-645 + both-modalities subjects, filtered to age <= 20 at each session

nki_trimmed_age_records = []
for ses in NKI_SESSIONS:
    ses_rows = nki_demo[nki_demo["session_id"] == ses].copy()
    ses_rows = ses_rows[ses_rows["participant_id"].isin(nki_both_per_session[ses])]
    ses_rows = ses_rows[(ses_rows["age"].notna()) & (ses_rows["age"] <= 20)]
    ses_rows["subject_session_id"] = ses_rows["participant_id"] + "-" + ses
    nki_trimmed_age_records.append(ses_rows[["subject_session_id", "age"]])

nki_trimmed_age_df = pd.concat(nki_trimmed_age_records, ignore_index=True)
nki_trimmed_age_pool = nki_trimmed_age_df["subject_session_id"].tolist()
print(f"NKI_Trimmed age pool (all sessions, FS & CPAC & age <= 20): {len(nki_trimmed_age_pool)}")
for ses in NKI_SESSIONS:
    n = sum(1 for s in nki_trimmed_age_pool if s.endswith(f"-{ses}"))
    print(f"  ses-{ses}: {n}")

NKI_Trimmed age pool (all sessions, FS & CPAC & age <= 20): 393
  ses-BAS1: 195
  ses-FLU1: 129
  ses-FLU2: 67
  ses-BAS2: 2


In [29]:
nki_trimmed_age_trainval, nki_trimmed_age_test = train_test_split(
    sorted(nki_trimmed_age_pool), test_size=AGE_TEST_SIZE, random_state=RANDOM_STATE
)

print(f"NKI_Trimmed age - trainval: {len(nki_trimmed_age_trainval)}, test: {len(nki_trimmed_age_test)}")

NKI_Trimmed age - trainval: 353, test: 40


---
## PNC

**Protocol**: acq-singleband, ses-PNC1, CC200 atlas, reg-36Parameter, PartialNilearn correlations

**Intersection**: FreeSurfer subjects & CPAC subjects (file on disk) & demographic label available

In [30]:
# --- PNC FreeSurfer subjects (no session subdir in PNC) ---
pnc_fs_dirs = glob.glob(rf"{dir_path}\PNC_FreeSurfer\freesurfer\sub-*")
pnc_fs_subjects = {os.path.basename(d).replace("sub-", "") for d in pnc_fs_dirs if os.path.isdir(d)}
print(f"PNC FreeSurfer subjects: {len(pnc_fs_subjects)}")

# --- PNC CPAC subjects (file must exist on disk) ---
cpac_pattern_pnc = rf"{dir_path}\PNC_CPAC\cpac_RBCv0\sub-*\ses-PNC1\func\*_task-rest_acq-singleband_atlas-CC200_space-MNI152NLin6ASym_reg-36Parameter_desc-PartialNilearn_correlations.tsv"
pnc_cpac_files = glob.glob(cpac_pattern_pnc)
pnc_cpac_subjects = set()
for f in pnc_cpac_files:
    basename = os.path.basename(f)
    sub_id = basename.split("_")[0].replace("sub-", "")
    pnc_cpac_subjects.add(sub_id)
print(f"PNC CPAC subjects (acq-singleband, file on disk): {len(pnc_cpac_subjects)}")

# --- Structural & Functional ---
pnc_both = pnc_fs_subjects & pnc_cpac_subjects
print(f"PNC FS & CPAC: {len(pnc_both)}")

PNC FreeSurfer subjects: 1439
PNC CPAC subjects (acq-singleband, file on disk): 1188
PNC FS & CPAC: 1188


In [31]:
# --- PNC demographics ---
# PNC subject IDs are integers - keep as string for consistent matching
pnc_demo = pd.read_csv(rf"{dir_path}\PNC_BIDS\study-PNC_desc-participants.tsv", sep="\t")
pnc_demo = pnc_demo.drop_duplicates(subset="participant_id")
pnc_demo["participant_id"] = pnc_demo["participant_id"].astype(str)

# --- PNC age pool ---
pnc_demo_both = pnc_demo[pnc_demo["participant_id"].isin(pnc_both)].copy()
pnc_age_pool = pnc_demo_both[pnc_demo_both["age"].notna()]["participant_id"].tolist()
print(f"PNC age pool (FS & CPAC & age not NaN): {len(pnc_age_pool)}")

# --- PNC psy pool ---
pnc_psy_pool = pnc_demo_both[pnc_demo_both["p_factor_mcelroy_harmonized_all_samples"].notna()]["participant_id"].tolist()
print(f"PNC psy pool  (FS & CPAC & psy  not NaN): {len(pnc_psy_pool)}")

PNC age pool (FS & CPAC & age not NaN): 1188
PNC psy pool  (FS & CPAC & psy  not NaN): 1187


In [32]:
# --- PNC splits ---
pnc_age_trainval, pnc_age_test = train_test_split(
    sorted(pnc_age_pool), test_size=AGE_TEST_SIZE, random_state=RANDOM_STATE
)
pnc_psy_trainval, pnc_psy_test = train_test_split(
    sorted(pnc_psy_pool), test_size=PSY_TEST_SIZE, random_state=RANDOM_STATE
)

print(f"PNC age  - trainval: {len(pnc_age_trainval)}, test: {len(pnc_age_test)}")
print(f"PNC psy  - trainval: {len(pnc_psy_trainval)}, test: {len(pnc_psy_test)}")

PNC age  - trainval: 1069, test: 119
PNC psy  - trainval: 795, test: 392


---
## CCNP

**Protocol**: NO_ACQ, ses-1, CC200 atlas, reg-36Parameter, PartialNilearn correlations

**Run strategy**: CCNP acquires two rest runs per session (run-01 and run-02) taken back-to-back with identical setup. Use run-01 per subject; fall back to run-02 if run-01 is missing on disk.

In [33]:
# --- CCNP FreeSurfer subjects ---
ccnp_fs_dirs = glob.glob(rf"{dir_path}\CCNP_FreeSurfer\freesurfer\sub-*")
ccnp_fs_subjects = {os.path.basename(d).replace("sub-", "") for d in ccnp_fs_dirs if os.path.isdir(d)}
print(f"CCNP FreeSurfer subjects: {len(ccnp_fs_subjects)}")

# --- CCNP CPAC subjects: run-01 first, fall back to run-02 ---
run01_files = glob.glob(
    rf"{dir_path}\CCNP_CPAC\cpac_RBCv0\sub-*\ses-1\func\*_task-rest_run-01_atlas-CC200_space-MNI152NLin6ASym_reg-36Parameter_desc-PartialNilearn_correlations.tsv"
)
run02_files = glob.glob(
    rf"{dir_path}\CCNP_CPAC\cpac_RBCv0\sub-*\ses-1\func\*_task-rest_run-02_atlas-CC200_space-MNI152NLin6ASym_reg-36Parameter_desc-PartialNilearn_correlations.tsv"
)

run01_subjects = {os.path.basename(f).split("_")[0].replace("sub-", "") for f in run01_files}
run02_subjects = {os.path.basename(f).split("_")[0].replace("sub-", "") for f in run02_files}

# Add run-02-only subjects (those without run-01)
ccnp_cpac_subjects = run01_subjects | (run02_subjects - run01_subjects)

print(f"CCNP CPAC run-01 subjects: {len(run01_subjects)}")
print(f"CCNP CPAC run-02 only (fallback): {len(run02_subjects - run01_subjects)}")
print(f"CCNP CPAC total (file on disk): {len(ccnp_cpac_subjects)}")

# --- Structural & Functional ---
ccnp_both = ccnp_fs_subjects & ccnp_cpac_subjects
print(f"CCNP FS & CPAC: {len(ccnp_both)}")

CCNP FreeSurfer subjects: 178
CCNP CPAC run-01 subjects: 149
CCNP CPAC run-02 only (fallback): 2
CCNP CPAC total (file on disk): 151
CCNP FS & CPAC: 151


In [34]:
# --- CCNP demographics ---
# participant_id is colornest001 format (no sub- prefix), single session/wave
ccnp_demo = pd.read_csv(rf"{dir_path}\CCNP_BIDS\study-CCNP_desc-participants.tsv", sep="\t")
ccnp_demo = ccnp_demo.drop_duplicates(subset="participant_id")
ccnp_demo["participant_id"] = ccnp_demo["participant_id"].astype(str)

# --- CCNP age pool ---
ccnp_demo_both = ccnp_demo[ccnp_demo["participant_id"].isin(ccnp_both)].copy()
ccnp_age_pool = ccnp_demo_both[ccnp_demo_both["age"].notna()]["participant_id"].tolist()
print(f"CCNP age pool (FS & CPAC & age not NaN): {len(ccnp_age_pool)}")

# --- CCNP psy pool ---
ccnp_psy_pool = ccnp_demo_both[ccnp_demo_both["p_factor_mcelroy_harmonized_all_samples"].notna()]["participant_id"].tolist()
print(f"CCNP psy pool  (FS & CPAC & psy  not NaN): {len(ccnp_psy_pool)}")

CCNP age pool (FS & CPAC & age not NaN): 151
CCNP psy pool  (FS & CPAC & psy  not NaN): 141


In [35]:
# --- CCNP splits ---
ccnp_age_trainval, ccnp_age_test = train_test_split(
    sorted(ccnp_age_pool), test_size=AGE_TEST_SIZE, random_state=RANDOM_STATE
)
ccnp_psy_trainval, ccnp_psy_test = train_test_split(
    sorted(ccnp_psy_pool), test_size=PSY_TEST_SIZE, random_state=RANDOM_STATE
)

print(f"CCNP age  - trainval: {len(ccnp_age_trainval)}, test: {len(ccnp_age_test)}")
print(f"CCNP psy  - trainval: {len(ccnp_psy_trainval)}, test: {len(ccnp_psy_test)}")

CCNP age  - trainval: 135, test: 16
CCNP psy  - trainval: 94, test: 47


---
## HBN

**Protocol**: NO_ACQ, exclude ses-HBNsiteSI (1.5T Avanto, TR=1.45s — incompatible with 3T Prisma sites), CC200 atlas, reg-36Parameter, PartialNilearn correlations

**Run strategy**: RU/CBIC/CUNY collected two 5-min rest runs (run-1, run-2) split from original 10-min scan to reduce head motion (Alexander et al., 2017, p.13). Runs are independent and same-session. Use run-1 per subject; fall back to run-2 if run-1 missing.

In [36]:
# --- HBN FreeSurfer subjects (no session subdir, NDAR* IDs) ---
hbn_fs_dirs = glob.glob(rf"{dir_path}\HBN_FreeSurfer\freesurfer\sub-*")
hbn_fs_subjects = {os.path.basename(d).replace("sub-", "") for d in hbn_fs_dirs if os.path.isdir(d)}
print(f"HBN FreeSurfer subjects: {len(hbn_fs_subjects)}")

# --- HBN CPAC subjects: exclude SI, NO_ACQ (no acq field), run-1 first, fall back to run-2 ---
# SI site (ses-HBNsiteSI) uses 1.5T Avanto TR=1.45s — excluded
# NO_ACQ = no 'acq-' token in filename

run1_files = [
    f for f in glob.glob(
        rf"{dir_path}\HBN_CPAC\cpac_RBCv0\sub-*\ses-HBNsite*\func"
        rf"\*_task-rest_run-1_atlas-CC200_space-MNI152NLin6ASym_reg-36Parameter_desc-PartialNilearn_correlations.tsv"
    )
    if "ses-HBNsiteSI" not in f and "_acq-" not in os.path.basename(f)
]

run2_files = [
    f for f in glob.glob(
        rf"{dir_path}\HBN_CPAC\cpac_RBCv0\sub-*\ses-HBNsite*\func"
        rf"\*_task-rest_run-2_atlas-CC200_space-MNI152NLin6ASym_reg-36Parameter_desc-PartialNilearn_correlations.tsv"
    )
    if "ses-HBNsiteSI" not in f and "_acq-" not in os.path.basename(f)
]

run1_subjects = {os.path.basename(f).split("_")[0].replace("sub-", "") for f in run1_files}
run2_subjects = {os.path.basename(f).split("_")[0].replace("sub-", "") for f in run2_files}

# run-1 first, fall back to run-2 if run-1 missing
hbn_cpac_subjects = run1_subjects | (run2_subjects - run1_subjects)

print(f"HBN CPAC run-1 subjects (non-SI, NO_ACQ): {len(run1_subjects)}")
print(f"HBN CPAC run-2 only fallback: {len(run2_subjects - run1_subjects)}")
print(f"HBN CPAC total (file on disk): {len(hbn_cpac_subjects)}")

# --- Structural & Functional ---
hbn_both = hbn_fs_subjects & hbn_cpac_subjects
print(f"HBN FS & CPAC: {len(hbn_both)}")

HBN FreeSurfer subjects: 1390
HBN CPAC run-1 subjects (non-SI, NO_ACQ): 899
HBN CPAC run-2 only fallback: 23
HBN CPAC total (file on disk): 922
HBN FS & CPAC: 844


In [37]:
# --- HBN demographics ---
# participant_id is NDAR format (no sub- prefix), one row per subject, no duplicates
hbn_demo = pd.read_csv(rf"{dir_path}\HBN_BIDS\study-HBN_desc-participants.tsv", sep="\t")
hbn_demo["participant_id"] = hbn_demo["participant_id"].astype(str)
# Exclude SI site at demographics level (consistent with CPAC exclusion)
hbn_demo = hbn_demo[hbn_demo["study_site"] != "HBNsiteSI"]

# --- HBN age pool ---
hbn_demo_both = hbn_demo[hbn_demo["participant_id"].isin(hbn_both)].copy()
hbn_age_pool = hbn_demo_both[hbn_demo_both["age"].notna()]["participant_id"].tolist()
print(f"HBN age pool (FS & CPAC & age not NaN): {len(hbn_age_pool)}")

# --- HBN psy pool ---
hbn_psy_pool = hbn_demo_both[hbn_demo_both["p_factor_mcelroy_harmonized_all_samples"].notna()]["participant_id"].tolist()
print(f"HBN psy pool  (FS & CPAC & psy  not NaN): {len(hbn_psy_pool)}")

HBN age pool (FS & CPAC & age not NaN): 844
HBN psy pool  (FS & CPAC & psy  not NaN): 758


In [38]:
# --- HBN splits ---
hbn_age_trainval, hbn_age_test = train_test_split(
    sorted(hbn_age_pool), test_size=AGE_TEST_SIZE, random_state=RANDOM_STATE
)
hbn_psy_trainval, hbn_psy_test = train_test_split(
    sorted(hbn_psy_pool), test_size=PSY_TEST_SIZE, random_state=RANDOM_STATE
)

print(f"HBN age  - trainval: {len(hbn_age_trainval)}, test: {len(hbn_age_test)}")
print(f"HBN psy  - trainval: {len(hbn_psy_trainval)}, test: {len(hbn_psy_test)}")

HBN age  - trainval: 759, test: 85
HBN psy  - trainval: 507, test: 251


---
## BHRC

**Protocol**: NO_ACQ, ses-1 only, run-1 preferred (run-2 fallback for 6 subjects where run-1 was removed from RBC), CC200 atlas, reg-36Parameter, PartialNilearn correlations

**Session note**: ses-2 exists for some subjects but has no separate age recorded; ses-1 is used exclusively. No subject has both run-1 and run-2 — run-2 exists only when run-1 failed QC and was removed. VARIANT acq groups are non-standard protocol deviations and excluded.

In [43]:
# --- BHRC FreeSurfer subjects (ses-1 only) ---
bhrc_fs_dirs = glob.glob(rf"{dir_path}\BHRC_FreeSurfer\freesurfer\sub-*_ses-1")
bhrc_fs_subjects = {os.path.basename(d).split("_")[0].replace("sub-", "") for d in bhrc_fs_dirs}
print(f"BHRC FreeSurfer subjects (ses-1): {len(bhrc_fs_subjects)}")

# --- BHRC CPAC subjects: ses-1, run-1 or run-2 (fallback for subjects where run-1 was removed), NO_ACQ, CC200 ---
bhrc_cpac_run1 = {
    os.path.basename(f).split("_")[0].replace("sub-", "")
    for f in glob.glob(
        rf"{dir_path}\BHRC_CPAC\cpac_RBCv0\sub-*\ses-1\func"
        r"\*_task-rest_run-1_atlas-CC200_space-MNI152NLin6ASym_reg-36Parameter_desc-PartialNilearn_correlations.tsv"
    )
    if "_acq-" not in os.path.basename(f)
}
bhrc_cpac_run2 = {
    os.path.basename(f).split("_")[0].replace("sub-", "")
    for f in glob.glob(
        rf"{dir_path}\BHRC_CPAC\cpac_RBCv0\sub-*\ses-1\func"
        r"\*_task-rest_run-2_atlas-CC200_space-MNI152NLin6ASym_reg-36Parameter_desc-PartialNilearn_correlations.tsv"
    )
    if "_acq-" not in os.path.basename(f)
}
bhrc_cpac_subjects = bhrc_cpac_run1 | bhrc_cpac_run2
print(f"BHRC CPAC subjects (ses-1, run-1 or run-2, NO_ACQ): {len(bhrc_cpac_subjects)}")
print(f"  run-1: {len(bhrc_cpac_run1)}  run-2 only: {len(bhrc_cpac_run2 - bhrc_cpac_run1)}")

# --- Structural & Functional ---
bhrc_both = bhrc_fs_subjects & bhrc_cpac_subjects
print(f"BHRC FS & CPAC: {len(bhrc_both)}")

BHRC FreeSurfer subjects (ses-1): 424
BHRC CPAC subjects (ses-1, run-1 or run-2, NO_ACQ): 288
  run-1: 282  run-2 only: 6
BHRC FS & CPAC: 288


In [44]:
# --- BHRC demographics ---
# participant_id is numeric (e.g. 10001), session_id column used to filter ses-1
bhrc_demo = pd.read_csv(rf"{dir_path}\BHRC_BIDS\study-BHRC_desc-participants.tsv", sep="\t")
bhrc_demo = bhrc_demo[bhrc_demo["session_id"] == 1].drop_duplicates(subset="participant_id")
bhrc_demo["participant_id"] = bhrc_demo["participant_id"].astype(str)

# --- BHRC age pool ---
bhrc_demo_both = bhrc_demo[bhrc_demo["participant_id"].isin(bhrc_both)].copy()
bhrc_age_pool = bhrc_demo_both[bhrc_demo_both["age"].notna()]["participant_id"].tolist()
print(f"BHRC age pool (FS & CPAC & age not NaN): {len(bhrc_age_pool)}")

# --- BHRC psy pool ---
bhrc_psy_pool = bhrc_demo_both[bhrc_demo_both["p_factor_mcelroy_harmonized_all_samples"].notna()]["participant_id"].tolist()
print(f"BHRC psy pool  (FS & CPAC & psy  not NaN): {len(bhrc_psy_pool)}")

BHRC age pool (FS & CPAC & age not NaN): 288
BHRC psy pool  (FS & CPAC & psy  not NaN): 285


In [45]:
# --- BHRC splits ---
bhrc_age_trainval, bhrc_age_test = train_test_split(
    sorted(bhrc_age_pool), test_size=AGE_TEST_SIZE, random_state=RANDOM_STATE
)
bhrc_psy_trainval, bhrc_psy_test = train_test_split(
    sorted(bhrc_psy_pool), test_size=PSY_TEST_SIZE, random_state=RANDOM_STATE
)

print(f"BHRC age  - trainval: {len(bhrc_age_trainval)}, test: {len(bhrc_age_test)}")
print(f"BHRC psy  - trainval: {len(bhrc_psy_trainval)}, test: {len(bhrc_psy_test)}")

BHRC age  - trainval: 259, test: 29
BHRC psy  - trainval: 190, test: 95


In [46]:
summary = pd.DataFrame({
    "Dataset": ["NKI", "NKI", "PNC", "PNC", "CCNP", "CCNP", "HBN", "HBN", "BHRC", "BHRC"],
    "Target": ["age", "psy", "age", "psy", "age", "psy", "age", "psy", "age", "psy"],
    "Total pool": [
        len(nki_age_pool), len(nki_psy_pool),
        len(pnc_age_pool), len(pnc_psy_pool),
        len(ccnp_age_pool), len(ccnp_psy_pool),
        len(hbn_age_pool), len(hbn_psy_pool),
        len(bhrc_age_pool), len(bhrc_psy_pool),
    ],
    "Trainval": [
        len(nki_age_trainval), len(nki_psy_trainval),
        len(pnc_age_trainval), len(pnc_psy_trainval),
        len(ccnp_age_trainval), len(ccnp_psy_trainval),
        len(hbn_age_trainval), len(hbn_psy_trainval),
        len(bhrc_age_trainval), len(bhrc_psy_trainval),
    ],
    "Test": [
        len(nki_age_test), len(nki_psy_test),
        len(pnc_age_test), len(pnc_psy_test),
        len(ccnp_age_test), len(ccnp_psy_test),
        len(hbn_age_test), len(hbn_psy_test),
        len(bhrc_age_test), len(bhrc_psy_test),
    ],
    "Test %": [
        f"{AGE_TEST_SIZE*100:.0f}%", f"{PSY_TEST_SIZE*100:.0f}%",
        f"{AGE_TEST_SIZE*100:.0f}%", f"{PSY_TEST_SIZE*100:.0f}%",
        f"{AGE_TEST_SIZE*100:.0f}%", f"{PSY_TEST_SIZE*100:.0f}%",
        f"{AGE_TEST_SIZE*100:.0f}%", f"{PSY_TEST_SIZE*100:.0f}%",
        f"{AGE_TEST_SIZE*100:.0f}%", f"{PSY_TEST_SIZE*100:.0f}%",
    ],
})
summary

,Dataset,Target,Total pool,Trainval,Test,Test %
0,NKI,age,1236,1112,124,10%
1,NKI,psy,166,111,55,33%
2,PNC,age,1188,1069,119,10%
3,PNC,psy,1187,795,392,33%
4,CCNP,age,151,135,16,10%
5,CCNP,psy,141,94,47,33%
6,HBN,age,844,759,85,10%
7,HBN,psy,758,507,251,33%
8,BHRC,age,288,259,29,10%
9,BHRC,psy,285,190,95,33%


---
## Save Splits to saved_variables.json

Subject IDs are saved as string lists. When loading in other notebooks, filter rows by `subject_id` (after casting to string) rather than using row indices - this avoids ordering bugs.

In [47]:
splits = {
    # NKI (all sessions: BAS1+FLU1+FLU2+BAS2, IDs as "{sub}-{ses}")
    "nki_age_trainval_subjects": [str(s) for s in nki_age_trainval],
    "nki_age_test_subjects":     [str(s) for s in nki_age_test],
    "nki_psy_trainval_subjects": [str(s) for s in nki_psy_trainval],
    "nki_psy_test_subjects":     [str(s) for s in nki_psy_test],
    # NKI_Trimmed (all sessions, age <= 20, IDs as "{sub}-{ses}")
    "nkitrimmed_age_trainval_subjects": [str(s) for s in nki_trimmed_age_trainval],
    "nkitrimmed_age_test_subjects":     [str(s) for s in nki_trimmed_age_test],
    # PNC
    "pnc_age_trainval_subjects": [str(s) for s in pnc_age_trainval],
    "pnc_age_test_subjects":     [str(s) for s in pnc_age_test],
    "pnc_psy_trainval_subjects": [str(s) for s in pnc_psy_trainval],
    "pnc_psy_test_subjects":     [str(s) for s in pnc_psy_test],
    # CCNP
    "ccnp_age_trainval_subjects": [str(s) for s in ccnp_age_trainval],
    "ccnp_age_test_subjects":     [str(s) for s in ccnp_age_test],
    "ccnp_psy_trainval_subjects": [str(s) for s in ccnp_psy_trainval],
    "ccnp_psy_test_subjects":     [str(s) for s in ccnp_psy_test],
    # HBN
    "hbn_age_trainval_subjects": [str(s) for s in hbn_age_trainval],
    "hbn_age_test_subjects":     [str(s) for s in hbn_age_test],
    "hbn_psy_trainval_subjects": [str(s) for s in hbn_psy_trainval],
    "hbn_psy_test_subjects":     [str(s) for s in hbn_psy_test],
    # BHRC
    "bhrc_age_trainval_subjects": [str(s) for s in bhrc_age_trainval],
    "bhrc_age_test_subjects":     [str(s) for s in bhrc_age_test],
    "bhrc_psy_trainval_subjects": [str(s) for s in bhrc_psy_trainval],
    "bhrc_psy_test_subjects":     [str(s) for s in bhrc_psy_test],
}

save_path = rf"{dir_path}\saved_variables.json"
with open(save_path, "w") as f:
    json.dump(splits, f, indent=2, ensure_ascii=True)

print(f"Saved to {save_path}")
for k, v in splits.items():
    print(f"  {k}: {len(v)} subjects")

Saved to C:\Users\Faruk\Code\CCIR_Project\saved_variables.json
  nki_age_trainval_subjects: 1112 subjects
  nki_age_test_subjects: 124 subjects
  nki_psy_trainval_subjects: 111 subjects
  nki_psy_test_subjects: 55 subjects
  nkitrimmed_age_trainval_subjects: 353 subjects
  nkitrimmed_age_test_subjects: 40 subjects
  pnc_age_trainval_subjects: 1069 subjects
  pnc_age_test_subjects: 119 subjects
  pnc_psy_trainval_subjects: 795 subjects
  pnc_psy_test_subjects: 392 subjects
  ccnp_age_trainval_subjects: 135 subjects
  ccnp_age_test_subjects: 16 subjects
  ccnp_psy_trainval_subjects: 94 subjects
  ccnp_psy_test_subjects: 47 subjects
  hbn_age_trainval_subjects: 759 subjects
  hbn_age_test_subjects: 85 subjects
  hbn_psy_trainval_subjects: 507 subjects
  hbn_psy_test_subjects: 251 subjects
  bhrc_age_trainval_subjects: 259 subjects
  bhrc_age_test_subjects: 29 subjects
  bhrc_psy_trainval_subjects: 190 subjects
  bhrc_psy_test_subjects: 95 subjects
